#### Importing Libraries

In [1]:
# Importing core libraries

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings

# Importing model selection utilities

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

# Importing classification models

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier

# Importing evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Importing label encoding

from sklearn.preprocessing import LabelEncoder

# Importing experiment tracking

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Configuring warnings

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


#### Loading the Processed Datasets

In [2]:
# Defining the processed data directory

processed_dir = Path("../data/processed")

# Loading processed training and testing features

X_train = pd.read_parquet(
    processed_dir / "X_train_processed.parquet"
)

X_test = pd.read_parquet(
    processed_dir / "X_test_processed.parquet"
)

# Loading training and testing target labels

y_train = pd.read_parquet(
    processed_dir / "y_train.parquet"
)["severity"]

y_test = pd.read_parquet(
    processed_dir / "y_test.parquet"
)["severity"]

# Displaying dataset dimensions

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

# Displaying target class distribution

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

# Verifying feature and target alignment

print("\nTraining Indices Aligned:", X_train.index.equals(y_train.index))
print("Testing Indices Aligned:", X_test.index.equals(y_test.index))

Training Features: (800, 13)
Testing Features: (200, 13)

Training Target: (800,)
Testing Target: (200,)

Training Target Distribution:
severity
Minor       286
Critical    267
Moderate    247
Name: count, dtype: int64

Testing Target Distribution:
severity
Minor       72
Critical    66
Moderate    62
Name: count, dtype: int64

Training Indices Aligned: True
Testing Indices Aligned: True


#### Observations on Loaded Datasets

- The training and testing datasets contain **800 and 200 records**, respectively, with 13 processed features.
- All three severity classes are represented in both datasets, with relatively balanced distributions.
- Feature and target indices are correctly aligned, confirming that the datasets are ready for model development.

#### Encoding the Target Variable

In [3]:
# Initializing and fitting the target label encoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Displaying class encoding

print("Target Class Mapping:")

for encoded_value, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name}: {encoded_value}")

# Displaying encoded target dimensions

print("\nEncoded Training Target:", y_train_encoded.shape)
print("Encoded Testing Target:", y_test_encoded.shape)

# Displaying encoded class distribution

print("\nEncoded Training Class Distribution:")
print(pd.Series(y_train_encoded).value_counts().sort_index())

print("\nEncoded Testing Class Distribution:")
print(pd.Series(y_test_encoded).value_counts().sort_index())

# Verifying encoding consistency

print("\nTraining Encoding Consistent:",
      np.array_equal(label_encoder.inverse_transform(y_train_encoded), y_train.to_numpy()))

print("Testing Encoding Consistent:",
      np.array_equal(label_encoder.inverse_transform(y_test_encoded), y_test.to_numpy()))

Target Class Mapping:
Critical: 0
Minor: 1
Moderate: 2

Encoded Training Target: (800,)
Encoded Testing Target: (200,)

Encoded Training Class Distribution:
0    267
1    286
2    247
Name: count, dtype: int64

Encoded Testing Class Distribution:
0    66
1    72
2    62
Name: count, dtype: int64

Training Encoding Consistent: True
Testing Encoding Consistent: True


In [4]:
# Defining artifact directories

processed_dir = Path("../data/processed")
preprocessor_dir = Path("../models/preprocessed")

preprocessor_dir.mkdir(parents=True, exist_ok=True)

# Saving encoded targets with original indices

pd.DataFrame(
    {"severity": y_train_encoded},
    index=y_train.index
).to_parquet(processed_dir / "y_train_encoded.parquet")

pd.DataFrame(
    {"severity": y_test_encoded},
    index=y_test.index
).to_parquet(processed_dir / "y_test_encoded.parquet")

# Saving the fitted label encoder

joblib.dump(
    label_encoder,
    preprocessor_dir / "label_encoder.joblib"
)

# Verifying saved artifacts

for file in [
    processed_dir / "y_train_encoded.parquet",
    processed_dir / "y_test_encoded.parquet",
    preprocessor_dir / "label_encoder.joblib"
]:
    print(f"{file.name}: {'Saved' if file.exists() else 'Missing'}")

y_train_encoded.parquet: Saved
y_test_encoded.parquet: Saved
label_encoder.joblib: Saved


#### Observations on Target Encoding

- The three severity classes were encoded numerically: **Critical (0), Minor (1), and Moderate (2)**.
- The encoded training and testing targets retain **800 and 200 records**, respectively.
- The encoded targets and fitted label encoder were saved for consistent use during model development, evaluation, and explainability.

#### Configuring Cross-Validation and Evaluation Metrics

In [5]:
# Configuring stratified cross-validation

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Defining classification evaluation metrics

scoring_metrics = {
    "Accuracy": "accuracy",
    "Precision_Macro": "precision_macro",
    "Recall_Macro": "recall_macro",
    "F1_Macro": "f1_macro"
}

# Displaying cross-validation configuration

print("Cross-Validation Strategy:", cv_strategy)
print("Number of Folds:", cv_strategy.get_n_splits())

print("\nEvaluation Metrics:")

for metric_name, scoring_method in scoring_metrics.items():
    print(f"{metric_name}: {scoring_method}")

# Verifying stratified class distribution across folds

print("\nClass Distribution Across Folds:")

for fold, (_, validation_idx) in enumerate(
    cv_strategy.split(X_train, y_train_encoded),
    start=1
):
    fold_labels = y_train_encoded[validation_idx]

    class_distribution = (
        pd.Series(fold_labels)
        .value_counts(normalize=True)
        .sort_index()
        .round(3)
    )

    print(f"\nFold {fold}:")
    print(class_distribution.to_dict())

Cross-Validation Strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Number of Folds: 5

Evaluation Metrics:
Accuracy: accuracy
Precision_Macro: precision_macro
Recall_Macro: recall_macro
F1_Macro: f1_macro

Class Distribution Across Folds:

Fold 1:
{0: 0.338, 1: 0.356, 2: 0.306}

Fold 2:
{0: 0.338, 1: 0.356, 2: 0.306}

Fold 3:
{0: 0.331, 1: 0.356, 2: 0.312}

Fold 4:
{0: 0.331, 1: 0.356, 2: 0.312}

Fold 5:
{0: 0.331, 1: 0.362, 2: 0.306}


#### Observations on Cross-Validation

- Stratified 5-Fold Cross-Validation was configured with shuffling and `random_state=42` for reproducibility.
- All five folds maintain consistent class distributions, with approximately **33% Critical, 36% Minor, and 31% Moderate** records.
- Macro F1-score was selected as the primary model comparison metric, supported by accuracy, macro precision, and macro recall.

#### Configuring MLflow Experiment Tracking

In [6]:
# Defining MLflow tracking configuration

project_root = Path("..").resolve()

mlflow_db_path = project_root / "mlflow.db"
mlflow_artifact_dir = project_root / "mlartifacts"

mlflow_artifact_dir.mkdir(parents=True, exist_ok=True)

# Configuring MLflow tracking URI

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_db_path.as_posix()}"
)

# Defining experiment name

experiment_name = "Manufacturing Quality Assurance"

# Creating or retrieving the MLflow experiment

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=mlflow_artifact_dir.as_uri()
    )
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(experiment_name)

# Displaying experiment configuration

print("Experiment Name:", experiment_name)
print("Experiment ID:", experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Artifact Directory:", mlflow_artifact_dir)
print("Experiment Status: Ready")

2026/10/20 23:07:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/20 23:07:01 INFO mlflow.store.db.utils: Updating database tables


Experiment Name: Manufacturing Quality Assurance
Experiment ID: 1
Tracking URI: sqlite:///D:/Projects/ML and Data Science/Manufacturing Quality Assurance/mlflow.db
Artifact Directory: D:\Projects\ML and Data Science\Manufacturing Quality Assurance\mlartifacts
Experiment Status: Ready


#### Observations on MLflow Configuration

- MLflow experiment tracking was configured successfully using a local SQLite database.
- The **Manufacturing Quality Assurance** experiment was created with Experiment ID **1**.
- The tracking database and artifact directory are ready to store model experiments, metrics, and trained models.

#### Developing the Baseline Model

In [7]:
# Initializing the baseline classifier

baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Evaluating the baseline using stratified cross-validation

baseline_cv_results = cross_validate(
    estimator=baseline_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

baseline_metrics = {
    metric: baseline_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting the baseline model on the complete training dataset

baseline_model.fit(X_train, y_train_encoded)

# Saving the trained baseline model

trained_models_dir = Path("../models/trained")
trained_models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    baseline_model,
    trained_models_dir / "dummy_classifier.joblib"
)

# Logging the baseline experiment to MLflow

with mlflow.start_run(run_name="Dummy Classifier"):

    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("cv_folds", 5)

    for metric, value in baseline_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "dummy_classifier.joblib"),
        artifact_path="model"
    )

# Creating the model comparison results

model_results = []

model_results.append({
    "Model": "Dummy Classifier",
    **baseline_metrics
})

# Displaying baseline performance

print("Baseline Model: Dummy Classifier")

print("\nCross-Validation Performance:")

for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")
display(pd.DataFrame(model_results))

Baseline Model: Dummy Classifier

Cross-Validation Performance:
Accuracy: 0.3575
Precision_Macro: 0.1192
Recall_Macro: 0.3333
F1_Macro: 0.1756

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Dummy Classifier,0.3575,0.119167,0.333333,0.175566


#### Observations on Baseline Model

- The Dummy Classifier achieved **35.75% accuracy** and a **Macro F1-score of 0.1756**.
- Its low Macro F1-score reflects its majority-class prediction strategy, which fails to identify the other two severity classes.
- These results establish the baseline that subsequent classification models should aim to outperform.

#### Developing the Logistic Regression Model

In [8]:
# Initializing the Logistic Regression classifier

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Evaluating Logistic Regression using stratified cross-validation

logistic_cv_results = cross_validate(
    estimator=logistic_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

logistic_metrics = {
    metric: logistic_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Logistic Regression on the complete training dataset

logistic_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    logistic_model,
    trained_models_dir / "logistic_regression.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Logistic Regression"):

    mlflow.log_params({
        "model": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in logistic_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "logistic_regression.joblib"),
        artifact_path="model"
    )

# Updating the model comparison results

model_results.append({
    "Model": "Logistic Regression",
    **logistic_metrics
})

# Displaying model performance

print("Model: Logistic Regression")

print("\nCross-Validation Performance:")

for metric, value in logistic_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")
display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Logistic Regression

Cross-Validation Performance:
Accuracy: 0.3612
Precision_Macro: 0.3642
Recall_Macro: 0.3594
F1_Macro: 0.3582

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Developing the Decision Tree Model

In [9]:
# Initializing the Decision Tree classifier

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

# Evaluating Decision Tree using stratified cross-validation

decision_tree_cv_results = cross_validate(
    estimator=decision_tree_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

decision_tree_metrics = {
    metric: decision_tree_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Decision Tree on the complete training dataset

decision_tree_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    decision_tree_model,
    trained_models_dir / "decision_tree.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Decision Tree"):

    mlflow.log_params({
        "model": "DecisionTreeClassifier",
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in decision_tree_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "decision_tree.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Decision Tree",
    **decision_tree_metrics
})

# Displaying model performance

print("Model: Decision Tree")

print("\nCross-Validation Performance:")

for metric, value in decision_tree_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Decision Tree

Cross-Validation Performance:
Accuracy: 0.3200
Precision_Macro: 0.3189
Recall_Macro: 0.3201
F1_Macro: 0.3187

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Decision Tree,0.32000,0.318900,0.320135,0.318748
2,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Decision Tree

- The Decision Tree achieved **32.00% accuracy** and a **Macro F1-score of 0.3187**.
- Logistic Regression outperformed Decision Tree, achieving a higher Macro F1-score of **0.3582**.
- Both models show relatively weak predictive performance, suggesting that defect severity may be difficult to distinguish using the available features.

#### Developing the Random Forest Model

In [10]:
# Initializing the Random Forest classifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Evaluating Random Forest using stratified cross-validation

random_forest_cv_results = cross_validate(
    estimator=random_forest_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

random_forest_metrics = {
    metric: random_forest_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Random Forest on the complete training dataset

random_forest_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    random_forest_model,
    trained_models_dir / "random_forest.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Random Forest"):

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 200,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in random_forest_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "random_forest.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Random Forest",
    **random_forest_metrics
})

# Displaying model performance

print("Model: Random Forest")

print("\nCross-Validation Performance:")

for metric, value in random_forest_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Random Forest

Cross-Validation Performance:
Accuracy: 0.2962
Precision_Macro: 0.2942
Recall_Macro: 0.2943
F1_Macro: 0.2934

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Decision Tree,0.32000,0.318900,0.320135,0.318748
2,Random Forest,0.29625,0.294175,0.294328,0.293363
3,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Random Forest

- Random Forest achieved **29.63% accuracy** and a **Macro F1-score of 0.2934**.
- It underperformed both Logistic Regression and Decision Tree, despite using an ensemble of decision trees.
- The results suggest that increasing model complexity has not improved severity classification performance so far.

#### Developing the Extra Trees Model

In [11]:
# Initializing the Extra Trees classifier

extra_trees_model = ExtraTreesClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Evaluating Extra Trees using stratified cross-validation

extra_trees_cv_results = cross_validate(
    estimator=extra_trees_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

extra_trees_metrics = {
    metric: extra_trees_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Extra Trees on the complete training dataset

extra_trees_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    extra_trees_model,
    trained_models_dir / "extra_trees.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Extra Trees"):

    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "n_estimators": 200,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in extra_trees_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "extra_trees.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Extra Trees",
    **extra_trees_metrics
})

# Displaying model performance

print("Model: Extra Trees")

print("\nCross-Validation Performance:")

for metric, value in extra_trees_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Extra Trees

Cross-Validation Performance:
Accuracy: 0.2863
Precision_Macro: 0.2844
Recall_Macro: 0.2849
F1_Macro: 0.2841

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Decision Tree,0.32000,0.318900,0.320135,0.318748
2,Random Forest,0.29625,0.294175,0.294328,0.293363
3,Extra Trees,0.28625,0.284362,0.284857,0.284067
4,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Extra Trees

- Extra Trees achieved **28.63% accuracy** and a **Macro F1-score of 0.2841**.
- It underperformed Random Forest, Decision Tree, and Logistic Regression.
- Tree-based models have shown limited effectiveness so far, suggesting weak predictive relationships between the available features and defect severity.

#### Developing the Gradient Boosting Model

In [12]:
# Initializing the Gradient Boosting classifier

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Evaluating Gradient Boosting using stratified cross-validation

gradient_boosting_cv_results = cross_validate(
    estimator=gradient_boosting_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

gradient_boosting_metrics = {
    metric: gradient_boosting_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Gradient Boosting on the complete training dataset

gradient_boosting_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    gradient_boosting_model,
    trained_models_dir / "gradient_boosting.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Gradient Boosting"):

    mlflow.log_params({
        "model": "GradientBoostingClassifier",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in gradient_boosting_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "gradient_boosting.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Gradient Boosting",
    **gradient_boosting_metrics
})

# Displaying model performance

print("Model: Gradient Boosting")

print("\nCross-Validation Performance:")

for metric, value in gradient_boosting_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Gradient Boosting

Cross-Validation Performance:
Accuracy: 0.3212
Precision_Macro: 0.3193
Recall_Macro: 0.3198
F1_Macro: 0.3175

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Decision Tree,0.32000,0.318900,0.320135,0.318748
2,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
3,Random Forest,0.29625,0.294175,0.294328,0.293363
4,Extra Trees,0.28625,0.284362,0.284857,0.284067
5,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Developing the XGBoost Model

In [13]:
# Initializing the XGBoost classifier

xgboost_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=2
)

# Evaluating XGBoost using stratified cross-validation

xgboost_cv_results = cross_validate(
    estimator=xgboost_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

xgboost_metrics = {
    metric: xgboost_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting XGBoost on the complete training dataset

xgboost_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    xgboost_model,
    trained_models_dir / "xgboost.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="XGBoost"):

    mlflow.log_params({
        "model": "XGBClassifier",
        "objective": "multi:softprob",
        "num_class": 3,
        "n_estimators": 150,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in xgboost_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "xgboost.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "XGBoost",
    **xgboost_metrics
})

# Displaying model performance

print("Model: XGBoost")

print("\nCross-Validation Performance:")

for metric, value in xgboost_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: XGBoost

Cross-Validation Performance:
Accuracy: 0.3250
Precision_Macro: 0.3246
Recall_Macro: 0.3225
F1_Macro: 0.3209

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,XGBoost,0.32500,0.324557,0.322450,0.320881
2,Decision Tree,0.32000,0.318900,0.320135,0.318748
3,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
4,Random Forest,0.29625,0.294175,0.294328,0.293363
5,Extra Trees,0.28625,0.284362,0.284857,0.284067
6,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on XGBoost

- XGBoost achieved **32.50% accuracy** and a **Macro F1-score of 0.3209**.
- It ranked second among standalone models but underperformed Logistic Regression (**0.3582 Macro F1**).
- Advanced boosting has not substantially improved severity classification, suggesting limited predictive information in the available features.

#### Saving Standalone Model Comparison

In [14]:
# Creating the standalone model comparison table

standalone_results = (
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

# Saving cross-validation results

standalone_results.to_csv(
    processed_dir / "standalone_model_comparison.csv",
    index=False
)

# Displaying the comparison

print("Standalone Model Comparison Saved Successfully.")

display(standalone_results.round(4))

Standalone Model Comparison Saved Successfully.


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.3612,0.3642,0.3594,0.3582
1,XGBoost,0.3250,0.3246,0.3225,0.3209
2,Decision Tree,0.3200,0.3189,0.3201,0.3187
3,Gradient Boosting,0.3212,0.3193,0.3198,0.3175
4,Random Forest,0.2962,0.2942,0.2943,0.2934
5,Extra Trees,0.2862,0.2844,0.2849,0.2841
6,Dummy Classifier,0.3575,0.1192,0.3333,0.1756


#### Tuning Logistic Regression

In [16]:
# Defining the Logistic Regression hyperparameter search space

logistic_param_distributions = {
    "C": np.logspace(-3, 2, 20),
    "class_weight": [None, "balanced"],
    "solver": ["lbfgs"],
}

# Initializing the Logistic Regression classifier

logistic_tuning_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Configuring randomized hyperparameter search

logistic_search = RandomizedSearchCV(
    estimator=logistic_tuning_model,
    param_distributions=logistic_param_distributions,
    n_iter=15,
    scoring="f1_macro",
    cv=cv_strategy,
    random_state=42,
    n_jobs=2,
    refit=True,
    return_train_score=True
)

# Fitting the hyperparameter search on training data

logistic_search.fit(X_train, y_train_encoded)

# Retrieving the best estimator and cross-validation score

tuned_logistic_model = logistic_search.best_estimator_

print("Model: Tuned Logistic Regression")

print("\nBest Hyperparameters:")
print(logistic_search.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(f"{logistic_search.best_score_:.4f}")

print("\nOriginal Logistic Regression Macro F1:")
print(f"{logistic_metrics['F1_Macro']:.4f}")

print("\nMacro F1 Improvement:")
print(
    f"{logistic_search.best_score_ - logistic_metrics['F1_Macro']:+.4f}"
)

Model: Tuned Logistic Regression

Best Hyperparameters:
{'solver': 'lbfgs', 'class_weight': None, 'C': np.float64(29.763514416313193)}

Best Cross-Validation Macro F1:
0.3612

Original Logistic Regression Macro F1:
0.3582

Macro F1 Improvement:
+0.0030


#### Observations on Logistic Regression Tuning

- Tuned Logistic Regression achieved a **Macro F1-score of 0.3612**, improving upon the original score of 0.3582.
- The best configuration used `C=29.7635`, `solver='lbfgs'`, and no class weighting.
- The improvement of **0.0030** is marginal, suggesting that hyperparameter tuning has limited impact on the available predictive signal.

#### Saving and Logging Tuned Logistic Regression

In [17]:
# Evaluating the tuned model using the selected hyperparameters

tuned_logistic_cv_results = cross_validate(
    estimator=tuned_logistic_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

tuned_logistic_metrics = {
    metric: tuned_logistic_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Saving the tuned model

joblib.dump(
    tuned_logistic_model,
    trained_models_dir / "tuned_logistic_regression.joblib"
)

# Logging the tuned model to MLflow

with mlflow.start_run(run_name="Tuned Logistic Regression"):

    mlflow.log_params({
        "model": "LogisticRegression",
        "C": float(logistic_search.best_params_["C"]),
        "solver": logistic_search.best_params_["solver"],
        "class_weight": str(logistic_search.best_params_["class_weight"]),
        "max_iter": 2000,
        "cv_folds": 5,
        "search_iterations": 15
    })

    for metric, value in tuned_logistic_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "tuned_logistic_regression.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Tuned Logistic Regression",
    **tuned_logistic_metrics
})

# Displaying tuned model performance

print("Model: Tuned Logistic Regression")

print("\nCross-Validation Performance:")

for metric, value in tuned_logistic_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nUpdated Model Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Tuned Logistic Regression

Cross-Validation Performance:
Accuracy: 0.3638
Precision_Macro: 0.3665
Recall_Macro: 0.3622
F1_Macro: 0.3612

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Updated Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Tuned Logistic Regression,0.36375,0.366465,0.362219,0.361235
1,Logistic Regression,0.36125,0.364212,0.359352,0.358197
2,XGBoost,0.32500,0.324557,0.322450,0.320881
3,Decision Tree,0.32000,0.318900,0.320135,0.318748
4,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
5,Random Forest,0.29625,0.294175,0.294328,0.293363
6,Extra Trees,0.28625,0.284362,0.284857,0.284067
7,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Tuned Logistic Regression

- Tuned Logistic Regression achieved **36.38% accuracy** and a **Macro F1-score of 0.3612**.
- It outperformed the original Logistic Regression model, although the improvement was marginal.
- It remains the strongest model so far, suggesting that simpler linear approaches may be better suited to the available features.

#### Tuning XGBoost

In [18]:
# Defining the XGBoost hyperparameter search space

xgboost_param_distributions = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [1, 5, 10]
}

# Initializing the XGBoost classifier

xgboost_tuning_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=2
)

# Configuring randomized hyperparameter search

xgboost_search = RandomizedSearchCV(
    estimator=xgboost_tuning_model,
    param_distributions=xgboost_param_distributions,
    n_iter=12,
    scoring="f1_macro",
    cv=cv_strategy,
    random_state=42,
    n_jobs=1,
    refit=True,
    return_train_score=True
)

# Fitting the hyperparameter search

xgboost_search.fit(X_train, y_train_encoded)

# Retrieving the best estimator

tuned_xgboost_model = xgboost_search.best_estimator_

# Displaying tuning results

print("Model: Tuned XGBoost")

print("\nBest Hyperparameters:")
print(xgboost_search.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(f"{xgboost_search.best_score_:.4f}")

print("\nOriginal XGBoost Macro F1:")
print(f"{xgboost_metrics['F1_Macro']:.4f}")

print("\nMacro F1 Improvement:")
print(
    f"{xgboost_search.best_score_ - xgboost_metrics['F1_Macro']:+.4f}"
)

Model: Tuned XGBoost

Best Hyperparameters:
{'subsample': 0.8, 'reg_lambda': 10, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

Best Cross-Validation Macro F1:
0.3350

Original XGBoost Macro F1:
0.3209

Macro F1 Improvement:
+0.0141


#### Observations on XGBoost Tuning

- Tuned XGBoost achieved a **Macro F1-score of 0.3350**, improving upon the original score of 0.3209.
- The optimized configuration favored shallower trees (`max_depth=2`) and stronger regularization (`reg_lambda=10`).
- Despite the improvement, XGBoost continued to underperform Tuned Logistic Regression.

#### Saving and Logging Tuned XGBoost

In [19]:
# Evaluating the tuned XGBoost model using cross-validation

tuned_xgboost_cv_results = cross_validate(
    estimator=tuned_xgboost_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

tuned_xgboost_metrics = {
    metric: tuned_xgboost_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Saving the tuned XGBoost model

joblib.dump(
    tuned_xgboost_model,
    trained_models_dir / "tuned_xgboost.joblib"
)

# Logging the tuned XGBoost experiment to MLflow

with mlflow.start_run(run_name="Tuned XGBoost"):

    mlflow.log_params({
        "model": "XGBClassifier",
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "random_state": 42,
        "cv_folds": 5,
        "search_iterations": 12,
        **{
            key: (
                value.item()
                if isinstance(value, np.generic)
                else value
            )
            for key, value in xgboost_search.best_params_.items()
        }
    })

    for metric, value in tuned_xgboost_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "tuned_xgboost.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Tuned XGBoost",
    **tuned_xgboost_metrics
})

# Displaying tuned model performance

print("Model: Tuned XGBoost")

print("\nCross-Validation Performance:")

for metric, value in tuned_xgboost_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nUpdated Model Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Tuned XGBoost

Cross-Validation Performance:
Accuracy: 0.3412
Precision_Macro: 0.3420
Recall_Macro: 0.3379
F1_Macro: 0.3350

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Updated Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Tuned Logistic Regression,0.36375,0.366465,0.362219,0.361235
1,Logistic Regression,0.36125,0.364212,0.359352,0.358197
2,Tuned XGBoost,0.34125,0.342020,0.337918,0.335022
3,XGBoost,0.32500,0.324557,0.322450,0.320881
4,Decision Tree,0.32000,0.318900,0.320135,0.318748
5,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
6,Random Forest,0.29625,0.294175,0.294328,0.293363
7,Extra Trees,0.28625,0.284362,0.284857,0.284067
8,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Tuned XGBoost

- Tuned XGBoost achieved **34.13% accuracy** and a **Macro F1-score of 0.3350**.
- Hyperparameter tuning improved Macro F1 by **0.0141**, outperforming the original XGBoost configuration.
- Tuned Logistic Regression remained the strongest model, with a Macro F1-score of **0.3612**.

#### Tuning Random Forest

In [20]:
# Defining the Random Forest hyperparameter search space

random_forest_param_distributions = {
    "n_estimators": [100, 150, 200],
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

# Initializing the Random Forest classifier

random_forest_tuning_model = RandomForestClassifier(
    random_state=42,
    n_jobs=2
)

# Configuring randomized hyperparameter search

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_tuning_model,
    param_distributions=random_forest_param_distributions,
    n_iter=10,
    scoring="f1_macro",
    cv=cv_strategy,
    random_state=42,
    n_jobs=1,
    refit=True,
    return_train_score=True
)

# Fitting the hyperparameter search

random_forest_search.fit(X_train, y_train_encoded)

# Retrieving the best estimator

tuned_random_forest_model = random_forest_search.best_estimator_

# Displaying tuning results

print("Model: Tuned Random Forest")

print("\nBest Hyperparameters:")
print(random_forest_search.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(f"{random_forest_search.best_score_:.4f}")

print("\nOriginal Random Forest Macro F1:")
print(f"{random_forest_metrics['F1_Macro']:.4f}")

print("\nMacro F1 Improvement:")
print(
    f"{random_forest_search.best_score_ - random_forest_metrics['F1_Macro']:+.4f}"
)

Model: Tuned Random Forest

Best Hyperparameters:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 3, 'class_weight': None}

Best Cross-Validation Macro F1:
0.3567

Original Random Forest Macro F1:
0.2934

Macro F1 Improvement:
+0.0634


#### Observations on Random Forest Tuning

- Tuned Random Forest achieved a **Macro F1-score of 0.3567**, improving upon the original score of 0.2934.
- The optimized configuration favored shallow trees (`max_depth=3`) and larger leaf sizes (`min_samples_leaf=4`).
- The improvement of **0.0634** brought Random Forest close to the leading Logistic Regression model.

#### Saving and Logging Tuned Random Forest

In [21]:
# Evaluating the tuned Random Forest using cross-validation

tuned_random_forest_cv_results = cross_validate(
    estimator=tuned_random_forest_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

tuned_random_forest_metrics = {
    metric: tuned_random_forest_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Saving the tuned Random Forest model

joblib.dump(
    tuned_random_forest_model,
    trained_models_dir / "tuned_random_forest.joblib"
)

# Logging the tuned Random Forest experiment to MLflow

with mlflow.start_run(run_name="Tuned Random Forest"):

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "random_state": 42,
        "cv_folds": 5,
        "search_iterations": 10,
        **{
            key: (
                value.item()
                if isinstance(value, np.generic)
                else str(value) if value is None else value
            )
            for key, value in random_forest_search.best_params_.items()
        }
    })

    for metric, value in tuned_random_forest_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "tuned_random_forest.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Tuned Random Forest",
    **tuned_random_forest_metrics
})

# Displaying tuned model performance

print("Model: Tuned Random Forest")

print("\nCross-Validation Performance:")

for metric, value in tuned_random_forest_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nUpdated Model Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Tuned Random Forest

Cross-Validation Performance:
Accuracy: 0.3700
Precision_Macro: 0.3730
Recall_Macro: 0.3640
F1_Macro: 0.3567

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Updated Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Tuned Logistic Regression,0.36375,0.366465,0.362219,0.361235
1,Logistic Regression,0.36125,0.364212,0.359352,0.358197
2,Tuned Random Forest,0.37000,0.373039,0.363995,0.356714
3,Tuned XGBoost,0.34125,0.342020,0.337918,0.335022
4,XGBoost,0.32500,0.324557,0.322450,0.320881
5,Decision Tree,0.32000,0.318900,0.320135,0.318748
6,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
7,Random Forest,0.29625,0.294175,0.294328,0.293363
8,Extra Trees,0.28625,0.284362,0.284857,0.284067
9,Dummy Classifier,0.35750,0.119167,0.333333,0.175566


#### Observations on Tuned Random Forest

- Tuned Random Forest achieved **37.00% accuracy** and a **Macro F1-score of 0.3567**.
- It recorded the highest accuracy among all models evaluated so far.
- Despite substantial improvement over the original Random Forest, its Macro F1-score remained slightly below Tuned Logistic Regression.

#### Developing the Soft Voting Ensemble

In [22]:
# Initializing the Soft Voting ensemble

soft_voting_model = VotingClassifier(
    estimators=[
        ("logistic_regression", tuned_logistic_model),
        ("random_forest", tuned_random_forest_model),
        ("xgboost", tuned_xgboost_model)
    ],
    voting="soft",
    n_jobs=2
)

# Evaluating Soft Voting using stratified cross-validation

soft_voting_cv_results = cross_validate(
    estimator=soft_voting_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

soft_voting_metrics = {
    metric: soft_voting_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting the ensemble on the complete training dataset

soft_voting_model.fit(X_train, y_train_encoded)

# Saving the trained ensemble

joblib.dump(
    soft_voting_model,
    trained_models_dir / "soft_voting.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Soft Voting Ensemble"):

    mlflow.log_params({
        "model": "VotingClassifier",
        "voting": "soft",
        "base_estimators": (
            "Tuned Logistic Regression, "
            "Tuned Random Forest, Tuned XGBoost"
        ),
        "cv_folds": 5
    })

    for metric, value in soft_voting_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "soft_voting.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Soft Voting Ensemble",
    **soft_voting_metrics
})

# Displaying ensemble performance

print("Model: Soft Voting Ensemble")

print("\nCross-Validation Performance:")

for metric, value in soft_voting_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nUpdated Model Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Soft Voting Ensemble

Cross-Validation Performance:
Accuracy: 0.3512
Precision_Macro: 0.3527
Recall_Macro: 0.3476
F1_Macro: 0.3442

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Updated Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Tuned Logistic Regression,0.36375,0.366465,0.362219,0.361235
1,Logistic Regression,0.36125,0.364212,0.359352,0.358197
2,Tuned Random Forest,0.37000,0.373039,0.363995,0.356714
3,Soft Voting Ensemble,0.35125,0.352696,0.347593,0.344226
4,Tuned XGBoost,0.34125,0.342020,0.337918,0.335022
5,XGBoost,0.32500,0.324557,0.322450,0.320881
6,Decision Tree,0.32000,0.318900,0.320135,0.318748
7,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
8,Random Forest,0.29625,0.294175,0.294328,0.293363
9,Extra Trees,0.28625,0.284362,0.284857,0.284067


#### Observations on Soft Voting Ensemble

- Soft Voting achieved **35.13% accuracy** and a **Macro F1-score of 0.3442**.
- It outperformed Tuned XGBoost but underperformed Tuned Logistic Regression and Tuned Random Forest.
- Combining model probabilities did not improve performance over the strongest standalone classifiers.

#### Developing the Stacking Ensemble

In [23]:
# Initializing the Stacking ensemble

stacking_model = StackingClassifier(
    estimators=[
        ("logistic_regression", tuned_logistic_model),
        ("random_forest", tuned_random_forest_model),
        ("xgboost", tuned_xgboost_model)
    ],
    final_estimator=LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=2
)

# Evaluating Stacking using stratified cross-validation

stacking_cv_results = cross_validate(
    estimator=stacking_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

stacking_metrics = {
    metric: stacking_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting the ensemble on the complete training dataset

stacking_model.fit(X_train, y_train_encoded)

# Saving the trained ensemble

joblib.dump(
    stacking_model,
    trained_models_dir / "stacking_ensemble.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Stacking Ensemble"):

    mlflow.log_params({
        "model": "StackingClassifier",
        "base_estimators": (
            "Tuned Logistic Regression, "
            "Tuned Random Forest, Tuned XGBoost"
        ),
        "final_estimator": "LogisticRegression",
        "stack_method": "predict_proba",
        "internal_cv_folds": 3,
        "evaluation_cv_folds": 5
    })

    for metric, value in stacking_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "stacking_ensemble.joblib"),
        artifact_path="model"
    )

# Updating model comparison results

model_results.append({
    "Model": "Stacking Ensemble",
    **stacking_metrics
})

# Displaying ensemble performance

print("Model: Stacking Ensemble")

print("\nCross-Validation Performance:")

for metric, value in stacking_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nUpdated Model Comparison:")

display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Stacking Ensemble

Cross-Validation Performance:
Accuracy: 0.3412
Precision_Macro: 0.3399
Recall_Macro: 0.3296
F1_Macro: 0.2924

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Updated Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Tuned Logistic Regression,0.36375,0.366465,0.362219,0.361235
1,Logistic Regression,0.36125,0.364212,0.359352,0.358197
2,Tuned Random Forest,0.37000,0.373039,0.363995,0.356714
3,Soft Voting Ensemble,0.35125,0.352696,0.347593,0.344226
4,Tuned XGBoost,0.34125,0.342020,0.337918,0.335022
5,XGBoost,0.32500,0.324557,0.322450,0.320881
6,Decision Tree,0.32000,0.318900,0.320135,0.318748
7,Gradient Boosting,0.32125,0.319318,0.319763,0.317466
8,Random Forest,0.29625,0.294175,0.294328,0.293363
9,Stacking Ensemble,0.34125,0.339866,0.329597,0.292371


#### Observations on Stacking Ensemble

- Stacking achieved **34.13% accuracy** and a **Macro F1-score of 0.2924**.
- It underperformed Soft Voting and all three tuned standalone models.
- The meta-model did not improve severity classification, indicating limited benefit from combining the available predictors.